In [ ]:
###### Create Engine #####
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
from sqlalchemy import text
import pandas as pd

# load environment variable from .env
load_dotenv()


db_url = os.getenv('DATABASE_URL')
engine = create_engine(db_url)


In [ ]:
##### Testing query performance issues ######
query = """
SELECT invoiceno, invoicedate
FROM oltp.orders
WHERE customerid = 17850;
"""

df = pd.read_sql(query, engine)
df.head(10)  

In [ ]:
query = """
SELECT transaction_id, quantity 
FROM oltp.order_items 
WHERE invoiceno = '536365' AND stockcode = '85123A';
"""

df = pd.read_sql(query, engine)
df.head(10)  

In [ ]:
print("Step 3: Implementing Performance Optimizations...")

with engine.connect() as conn:
    # Single-Column Indexes
    # Creating indexes on frequently queried columns to prevent full table scans
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_orders_invoicedate ON oltp.orders(invoicedate);"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_orders_customerid ON oltp.orders(customerid);"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_products_stockcode ON oltp.products(stockcode);"))
    
    # Composite Indexes
    # Creating a composite index on the order_items table to speed up JOINs 
    # between orders, order_items, and products
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_orderitems_invoice_stock ON oltp.order_items(invoiceno, stockcode);"))
    
    conn.commit()
    print("Indexes successfully created! Query performance optimized.")


In [ ]:
##### Rebuilding the Orders Table with Partitions ######

print("Rebuilding Orders table with Year/Month Partitions...")

with engine.connect() as conn:
    # Drop the existing tables (Order_Items first due to foreign key dependency)
    conn.execute(text("DROP TABLE IF EXISTS oltp.order_items CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS oltp.orders CASCADE;"))
    
    # Recreating Orders as a PARTITIONED table
    # We include invoicedate in the Primary Key, which is required for partitioning in Postgres
    conn.execute(text("""
        CREATE TABLE oltp.orders (
            invoiceno VARCHAR(50),
            invoicedate TIMESTAMP,
            customerid INT REFERENCES oltp.customers(customerid),
            PRIMARY KEY (invoiceno, invoicedate)
        ) PARTITION BY RANGE (invoicedate);
    """))
    
    # Create individual partitions (e.g., for late 2010 and early 2011)
    # EDA shows The UCI dataset spans Dec 2010 to Dec 2011
    conn.execute(text("""
        CREATE TABLE oltp.orders_2010_12 PARTITION OF oltp.orders 
        FOR VALUES FROM ('2010-12-01') TO ('2011-01-01');
    """))

    # create first-half of 2011 orders
    conn.execute(text("""
        CREATE TABLE oltp.orders_2011_h1 PARTITION OF oltp.orders 
        FOR VALUES FROM ('2011-01-01') TO ('2011-07-01');
    """))

    # create second-half of 2011 orders
    conn.execute(text("""
        CREATE TABLE oltp.orders_2011_h2 PARTITION OF oltp.orders 
        FOR VALUES FROM ('2011-07-01') TO ('2012-01-01');
    """))
    
    # Recreate Order_Items to reference the new composite Primary Key
    conn.execute(text("""
        CREATE TABLE oltp.order_items (
            transaction_id INT PRIMARY KEY,
            invoiceno VARCHAR(50),
            invoicedate TIMESTAMP,
            stockcode VARCHAR(50) REFERENCES oltp.products(stockcode),
            quantity INT CHECK (quantity <> 0),
            FOREIGN KEY (invoiceno, invoicedate) REFERENCES oltp.orders(invoiceno, invoicedate)
        );
    """))
    
    conn.commit()
    print("Partitioned schema created successfully.")


In [ ]:

print("Reloading data into partitioned schema...")

with engine.connect() as conn:
    # Reload Orders
    conn.execute(text("""
        INSERT INTO oltp.orders (invoiceno, invoicedate, customerid)
        SELECT DISTINCT invoiceno, invoicedate, customerid
        FROM retail_data_clean
        ON CONFLICT (invoiceno, invoicedate) DO NOTHING;
    """))
    
    # Reload Order_Items (now including invoicedate for the FK)
    conn.execute(text("""
        INSERT INTO oltp.order_items (transaction_id, invoiceno, invoicedate, stockcode, quantity)
        SELECT transaction_id, invoiceno, invoicedate, stockcode, quantity
        FROM retail_data_clean
        WHERE quantity <> 0
        ON CONFLICT (transaction_id) DO NOTHING;
    """))
    
    conn.commit()
    print("Data successfully loaded into partitioned OLTP schema!")